# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

MSSV     = "2A202602940"
REPO_URL = "https://github.com/tam253211-a11y/K4-Track4-DangHuuTam-2A202602940-Neuralnetwork.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Repo nằm trên Google Drive để figures/, results/ không mất khi Colab ngắt kết nối.
    from google.colab import drive
    drive.mount("/content/drive")
    COLAB_REPO = "/content/drive/MyDrive/K4-Track4-DangHuuTam-2A202602940-Neuralnetwork"
    if not os.path.isdir(COLAB_REPO):
        subprocess.run(["git", "clone", REPO_URL, COLAB_REPO], check=True)
    else:
        subprocess.run(["git", "-C", COLAB_REPO, "pull", "--ff-only"], check=False)
    # Làm việc tại submission_<MSSV>/code/ để REPO_ROOT = "../.." và OUT_DIR = ".." đúng như khi chạy local.
    os.chdir(f"{COLAB_REPO}/submission_{MSSV}/code")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
assert os.path.isfile(f"{REPO_ROOT}/scripts/split_data.py"), f"REPO_ROOT sai: {os.path.abspath(REPO_ROOT)}"
assert os.path.basename(os.path.abspath(OUT_DIR)) == f"submission_{MSSV}", f"OUT_DIR sai: {os.path.abspath(OUT_DIR)}"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("cwd      :", os.getcwd())
print("REPO_ROOT:", os.path.abspath(REPO_ROOT))
print("OUT_DIR  :", os.path.abspath(OUT_DIR))
print("torch    :", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU      :", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Chia train/eval theo metadata (script tự kiểm tra số dòng, row_id, số mẫu mỗi tập). Chạy từ gốc repo.
r = subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True, capture_output=True, text=True)
print(r.stdout)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

# --- Kích thước và kiểu
assert len(data["X_tr"]) == 371_847 and len(data["X_val"]) == 92_962 and len(data["X_eval"]) == 116_203
assert all(data[k].dtype == torch.float32 and data[k].shape[1] == 54 for k in ("X_tr", "X_val", "X_eval"))
assert all(data[k].dtype == torch.int64 for k in ("y_tr", "y_val", "y_eval"))

# --- Tỉ lệ lớp ở train / val / eval (phân tầng -> gần như bằng nhau)
print("\nlớp   train(%)   val(%)   eval(%)")
props = {k: torch.bincount(data[k], minlength=7).double() / len(data[k]) * 100 for k in ("y_tr", "y_val", "y_eval")}
for c in range(7):
    print(f"{c:>3d}   {props['y_tr'][c]:7.3f}  {props['y_val'][c]:7.3f}  {props['y_eval'][c]:7.3f}")

# --- Thống kê chuẩn hoá: 10 cột số trên X_tr ~ 0 / 1; 44 cột nhị phân vẫn chỉ có 0/1
num = data["X_tr"][:, :10].double()
print("\nmean 10 cột số trên X_tr:", num.mean(0).cpu().numpy().round(4))
print("std  10 cột số trên X_tr:", num.std(0).cpu().numpy().round(4))
assert num.mean(0).abs().max() < 1e-3 and (num.std(0) - 1).abs().max() < 1e-2
assert torch.isin(data["X_tr"][:, 10:], torch.tensor([0.0, 1.0], device=device)).all()
# val/eval chỉ được BIẾN ĐỔI bằng mean/std của train nên trung bình của chúng không đúng bằng 0 — đó là bình thường
print("mean |cột số| trên X_val:", round(data["X_val"][:, :10].mean(0).abs().max().item(), 4),
      "| trên X_eval:", round(data["X_eval"][:, :10].mean(0).abs().max().item(), 4))

# --- eval_row_id vẫn gắn đúng thứ tự mẫu eval (cùng thứ tự với eval.npz, dùng để ghi predictions_eval.csv)
_ev = np.load(f"{REPO_ROOT}/data/processed/eval.npz")
assert np.array_equal(data["eval_row_id"], _ev["row_id"]) and np.array_equal(data["y_eval"].cpu().numpy(), _ev["y"])
print("eval_row_id:", data["eval_row_id"][:5], "... (", len(data["eval_row_id"]), "dòng, khớp eval.npz )")

# --- Lô cuối: giữ lại (drop_last=False). 371 847 = 726*512 + 135 -> 727 bước/epoch
n_batches = sum(1 for _ in iterate_batches(data["X_tr"], data["y_tr"], 512, shuffle=False))
print(f"\nSố lô/epoch với batch 512: {n_batches} (lô cuối {len(data['X_tr']) % 512} mẫu)")
print(f"Mốc 'đoán đa số' trên val: {data['majority_val_acc']:.4f} (GUIDE: ≈ 0.4876)")

**Nhận xét Part 0:**
- Val tách từ train (20%, phân tầng, seed 42): 371 847 train / 92 962 val; eval 116 203 giữ nguyên theo metadata. Seed tách val cố định 42 cho mọi thí nghiệm; khi đo nhiễu chỉ đổi seed huấn luyện.
- **Vì sao mean/std chỉ tính trên X_tr:** val dùng để chọn lr/cấu hình/epoch, eval dùng để chấm điểm cuối. Nếu thống kê của chúng lọt vào bước chuẩn hoá thì thông tin về dữ liệu "chưa thấy" đã đi vào pipeline (rò rỉ), số đo trên val/eval không còn là ước lượng trung thực cho dữ liệu mới. Val và eval chỉ được *áp dụng* phép biến đổi đã fit trên train, nên mean của chúng gần 0 nhưng không đúng bằng 0.
- Chỉ chuẩn hoá 10 cột số; 44 cột one-hot giữ 0/1 vì đã cùng thang đo và trừ mean sẽ làm mất tính thưa.
- **Lô cuối:** giữ lại lô cuối nhỏ hơn (135 mẫu với batch 512), 727 bước/epoch, mọi mẫu train đều được dùng mỗi epoch. Loss lấy trung bình trên lô nên thang đo không đổi.
- Mốc thấp nhất: luôn đoán lớp 1 cho val accuracy ≈ 0,4876; mọi mô hình phải vượt mốc này.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# TODO 1: model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device); assert count_params(model) == EXPECTED_PARAMS[(256,128)]
# TODO 2: cho một lô ngẫu nhiên (8, 54) chạy qua; in shape; assert logits.shape == (8, 7)
# TODO 3: loss bước 0 trên val (eval mode) ≈ ln 7 ≈ 1.946  -> in giá trị đo và nhận xét
# TODO 4: quá khớp 20 mẫu (tắt dropout, vài trăm bước) -> loss về gần 0; vẽ đường cong loss
# TODO 5: sau một lần backward, in grad norm của từng tham số; assert tất cả khác None và khác 0

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# TODO: chọn lr cho baseline bằng VAL (chạy vài giá trị, ví dụ quanh 0.01 -> 0.1); ghi bảng lr -> val_macro_f1
# TODO: cfg = {**DEFAULT_CFG, "lr": <lr đã chọn>}
# TODO: chạy baseline với >= 2 seed (khuyến khích 3): result = run_experiment({**cfg, "seed": s, "exp_id": f"base-s{s}"}, data)
# TODO: với mỗi result: save_result(result, f"{OUT_DIR}/results"); plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
# TODO: tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed -> ngưỡng nhiễu 2σ

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary